In [1]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [2]:
from testgen.prompts import Sensors

labels = Sensors.split("\n")
for i, sensor in enumerate(labels):
    labels[i] = sensor[sensor.find("(")+1: sensor.find(")")].strip()
    
labels = np.array(labels)

labels

array(['Acc', 'WSA', 'WS', 'YR', 'ST'], dtype='<U3')

In [3]:
# read results file
base_path = Path().cwd()
results_path = base_path.parent / "results"

available_results = list(results_path.glob("bulk*.json"))

# drop file with lower accuracy if multiple are found
available_results_dict = {}

for i, f in enumerate(available_results):
    t_ = f.name.split("_acc-")
    if available_results_dict.get(t_[0]) is None:
        available_results_dict[t_[0]] = (i, float(t_[1].split("_")[0]))
    else:
        if available_results_dict[t_[0]][1] < float(t_[1].split("_")[0]):
            del available_results[available_results_dict[t_[0]][0]]
            available_results_dict[t_[0]] = (i, float(t_[1].split("_")[0]))
        

available_results

[PosixPath('/mnt/d/scripts/hil/hil-test-case-gen/results/bulk-3_gpt-4o-mini_n-1_acc-0.828_10.29.2024-09:36:37.json'),
 PosixPath('/mnt/d/scripts/hil/hil-test-case-gen/results/bulk-5_gpt-4o-mini_n-1_acc-0.854_10.29.2024-09:31:56.json'),
 PosixPath('/mnt/d/scripts/hil/hil-test-case-gen/results/bulk-8_gpt-4o-mini_n-1_acc-0.832_10.29.2024-09:38:32.json')]

In [27]:
def calc_scores(df):
    df_scores = pd.DataFrame(
        columns=["sensor", "accuracy", "precision", "recall", "f1"]
    )

    y_true = df["true_label"]
    y_pred = df["pred_label"]
    unique_labels = np.unique(y_true)
    unique_labels.sort()

    accuracy_score_ = round(accuracy_score(y_true, y_pred), 2)
    precision_score_ = round(
        precision_score(y_true, y_pred, average="weighted", labels=unique_labels), 2
    )
    recall_score_ = round(
        recall_score(y_true, y_pred, average="weighted", labels=unique_labels), 2
    )
    f1_score_ = round(
        f1_score(y_true, y_pred, average="weighted", labels=unique_labels), 2
    )

    df_scores.loc[df_scores.shape[0] + 1] = [
        "All",
        accuracy_score_,
        precision_score_,
        recall_score_,
        f1_score_,
    ]

    for label in unique_labels:
        t_ = df[df["true_label"] == label]

        y_true_ = t_["true_label"]
        y_pred_ = t_["pred_label"]

        accuracy_score_ = round(accuracy_score(y_true_, y_pred_), 2)
        precision_score_ = round(
            precision_score(y_true_, y_pred_, average="weighted", labels=[label]), 2
        )
        recall_score_ = round(
            recall_score(y_true_, y_pred_, average="weighted", labels=[label]), 2
        )
        f1_score_ = round(
            f1_score(y_true_, y_pred_, average="weighted", labels=[label]), 2
        )

        df_scores.loc[df_scores.shape[0] + 1] = [
            label,
            accuracy_score_,
            precision_score_,
            recall_score_,
            f1_score_,
        ]

    return df_scores.set_index("sensor")

In [29]:
def analyze(filename):

    type_, model_name, number_examples, *_ = filename.stem.split("_")
    number_examples = int(number_examples.split("-")[-1])

    file_under_investigation = results_path / filename
    with file_under_investigation.open("r") as f:
        data = json.load(f)
        
    # split responses and general stats
    responses = pd.DataFrame()
    for res in data["responses"]:
        responses = pd.concat([responses, pd.DataFrame(res)])
    responses.set_index("idx", inplace=True)
    # del data["responses"]

    # collect stats per experiment
    stats = pd.DataFrame({k: [v]for k,v in data.items()})
    stats.insert(0, "type", type_)
    stats.insert(1, "model_name", model_name)
    stats.insert(2, "number_examples", number_examples)

    
    return data, stats, responses

In [30]:
filename = available_results[0]

data, stats_, responses = analyze(filename)

In [57]:
for i, d in enumerate(data["responses"]):

    vectors = [parse_result(r) for r in d["ai_response"].split("\n")]
    true_v = d["true_vector"]
    acc = [True if gt == v else False for gt, v in zip(true_v, vectors)]
    if not (acc == d["accuracy"]):
        print(i)
        data["responses"][i]["pred_vector"] = vectors
        data["responses"][i]["accuracy"] = acc

In [66]:
responses[responses["pred_vector"].str.len() != 11]

,requirement,true_vector,ai_response,pred_vector,response_time,accuracy,completion_tokens,prompt_tokens,total_tokens
idx,,,,,,,,,
136,The steering system should adapt the steering ...,"[0,1,0,0,0]","Vector [1]: [0,0,0,0,0] \nVector [2]: [0,0,0,...",[1],0.607839,False,48,877,925
56,"Upon detection of a fault condition, the syste...","[0,1,0,0,0]","Vector [1]: [0,0,0,0,0] \nVector [2]: [0,0,0,...",[2],0.607839,False,48,877,925
159,The power steering system must perform reliabl...,"[0,0,0,0,1]","Vector [1]: [0,0,0,0,0] \nVector [2]: [0,0,0,...",[3],0.607839,False,48,877,925


In [8]:
import re

In [14]:
t_ = responses[responses["pred_vector"].str.len() != 11]["ai_response"].to_list()

# t_ = [
#     'Vector 1: [0,0,0,0,0]  \nVector 2: [0,0,0,0,0]  \nVector 3: [0,0,0,0,0]',
#     'Vector [1]: [0,0,0,0,0]  \nVector [2]: [0,0,0,0,0]  \nVector [3]: [0,0,0,0,0]  '
# ]
# t_

In [26]:
def parse_result(res):
    """Parse the LLM result"""
    # pattern = r"\[.*?,.*?\]"
    pattern = r"\[[0-9,]*?,.*?\]"
    vec = re.findall(pattern, res)[0]
    return vec.replace(" ", "")

# pattern = r"\[([0-9,]+)\]"
pattern = r"\[[0-9,]*?,.*?\]"

for res in t_:
    vectors = [parse_result(v) for v in res.split("\n")]
    print(vectors)

['[0,0,0,0,0]', '[0,0,0,0,0]', '[0,0,0,0,0]']
['[0,0,0,0,0]', '[0,0,0,0,0]', '[0,0,0,0,0]']
['[0,0,0,0,0]', '[0,0,0,0,0]', '[0,0,0,0,0]']


In [43]:
responses[responses["pred_vector"].str.len() != 11]["ai_response"].to_list()

['Vector [1]: [0,0,0,0,0]  \nVector [2]: [0,0,0,0,0]  \nVector [3]: [0,0,0,0,0]  ',
 'Vector [1]: [0,0,0,0,0]  \nVector [2]: [0,0,0,0,0]  \nVector [3]: [0,0,0,0,0]  ',
 'Vector [1]: [0,0,0,0,0]  \nVector [2]: [0,0,0,0,0]  \nVector [3]: [0,0,0,0,0]  ']

In [ ]:

# collect label names for predictions and ground truth
responses["true_label"] = responses["true_vector"].map(lambda x: " & ".join(labels[np.array(x[1:-1].split(",")).astype(bool)]) )
responses["pred_label"] = responses["pred_vector"].map(lambda x: " & ".join(labels[np.array(x[1:-1].split(",")).astype(bool)]) )

# accuracy per sensor
summarize_ = responses.groupby(["true_label"]).aggregate({
    "accuracy": "sum",
    "true_label": "count"
})

summarize_.columns = ["true", "total"]
summarize_["false"] = summarize_["total"] - summarize_["true"]


all_vals = summarize_.sum(axis=0).values.tolist()
summarize_.loc["All"] = all_vals

summarize_.insert(0, "type", type_)
summarize_.insert(1, "model_name", model_name)
summarize_.insert(2, "number_examples", number_examples)

df_scores = calc_scores(responses)

summarize_ = summarize_.merge(df_scores, left_index=True, right_index=True).reset_index(names="sensors")

In [17]:
def plot_summarize(summarize_):
    ax = summarize_["accuracy"].plot.bar(
        title="Accuracy per Sensor",
        xlabel="",
    )

    _ = ax.bar_label(ax.containers[0])

In [20]:
# stats = pd.DataFrame()
# summarize = pd.DataFrame()

# for filename in available_results:
#     stats_, summarize_ = analyze(filename)
    
#     stats = pd.concat([stats, stats_])
#     summarize = pd.concat([summarize, summarize_])
    

# stats.drop(columns="examples", inplace=True)
# stats.set_index("number_examples", inplace=True)

# summarize.reset_index(drop=True, inplace=True)